# S04 -- užklausos plėtimas ir vertimas

**A kelias (be programavimo).** Paleiskite 1-3 langelius iš eilės. Matysite, kaip pradinė užklausa išskaidoma į koncepcijas ir sinonimus ir kaip ta pati paieška užrašoma PubMed, OpenAlex ir Scopus sintakse. Be rakto galite ranka keisti koncepcijas ir sinonimus (2 langelis) ir matyti, kaip keičiasi vertimas. Užklausos laukelis (SEED) naudojamas tik įjungus USE_LIVE: tada tikras DKM išplečia JŪSŲ užklausą (reikia savo OpenAI rakto, kreipinys apmokamas).

**B kelias (programuotojams)** -- žemiau, po demonstracija: patys įgyvendinkite funkcijas.

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/rimraul/slr-automation.git"
REPO_DIR = REPO_URL.rsplit("/", 1)[-1].removesuffix(".git")
if "google.colab" in sys.modules and not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
if os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR)
sys.path.insert(0, os.getcwd())
for req in ['examples/chapter/requirements-colab.txt']:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", req],
                       capture_output=True, text=True)
    if r.returncode:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise SystemExit("pip install nepavyko (zr. klaida aukščiau): " + req)
print("Paruosta:", os.getcwd())

In [ ]:
#@title 1. Pasirinkite užklausą ir duomenų bazes
SEED = "large language models for systematic review screening"  #@param {type:"string"}
PUBMED = True  #@param {type:"boolean"}
OPENALEX = True  #@param {type:"boolean"}
SCOPUS = True  #@param {type:"boolean"}
# Pagal nutylėjimą DKM NEnaudojamas (nemokama): koncepcijas redaguojate patys (2 langelis),
# o SEED čia tik pavyzdys. SEED veikia tik kai USE_LIVE įjungtas.
# Pažymėję USE_LIVE, DKM išplės JŪSŲ užklausą (reikia rakto Colab "Secrets";
# kreipinys apmokamas, paprastai kainuoja centus).
USE_LIVE = False  #@param {type:"boolean"}

In [ ]:
#@title 2. Redaguokite koncepcijas ir sinonimus (kai USE_LIVE išjungtas)
# Kiekviena eilutė -- viena koncepcija; sąrašas -- jos sinonimai.
# Pridėkite, ištrinkite ar pakeiskite terminus ir paleiskite 3 langelį iš naujo.
CONCEPTS = {
    'large language models': ['large language model', 'LLM', 'GPT', 'generative AI'],
    'systematic review': ['systematic review', 'evidence synthesis', 'scoping review'],
    'screening': ['title and abstract screening', 'study selection', 'citation screening'],
}

In [ ]:
#@title 3. Išplėsti ir išversti
from examples.chapter.s04_query_expansion import solution as s4

if USE_LIVE:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("Kreipinys į DKM (apmokamas). Užklausa:", SEED)
    expansion = s4.expand_query(SEED)  # tikras DKM kreipinys
else:
    expansion = s4.QueryExpansion(
        concepts=[s4.Concept(label=k, terms=v) for k, v in CONCEPTS.items()],
        boolean_query=" AND ".join("(" + " OR ".join(v) + ")" for v in CONCEPTS.values()))

print("KONCEPCIJOS IR SINONIMAI")
for c in expansion.concepts:
    print(f"  {c.label}: {', '.join(c.terms)}")

chosen = [d for d, on in (("pubmed", PUBMED), ("openalex", OPENALEX), ("scopus", SCOPUS)) if on]
print("\nVERTIMAS Į DUOMENŲ BAZIŲ SINTAKSĘ (be DKM, grynos taisyklės)")
for db in chosen:
    print(f"\n[{db}]\n{s4.translate_for_database(expansion, db)}")
print("\nSkirtumai: PubMed prideda [tiab] prie kiekvieno termino; OpenAlex -- be lauko žymų; "
      "Scopus apgaubia visą išraišką TITLE-ABS-KEY(...).")
print("Prisiminkite: DKM pasiūlytą užklausą visada turi peržiūrėti žmogus.")

In [ ]:
#@title (nebūtina) Ką DKM gauna iš tikrųjų?
from examples.chapter.s04_query_expansion import solution as s4
print("SISTEMOS INSTRUKCIJA:\n" + s4._SYSTEM)
print("\nVARTOTOJO ŽINUTĖ:\n" + SEED)

---
## B kelias: parašykite kodą patys

Toliau esantis pratimas skirtas tiems, kas nori išmokti programuoti.

# S04 pratimas

```
S4 pratimas -- paieškos užklausos plėtimas su DKM (LLM).

Užduotis
--------
1. Įgyvendinkite ``expand_query(seed, *, call_llm)`` -- nusiųskite pradinę
   užklausą DKM, gaukite JSON su koncepcijomis ir jų sinonimais, patikrinkite
   jį ``QueryExpansion`` modeliu. Iškelkite ``ValueError``, jei JSON netinkamas
   arba nėra nė vienos koncepcijos.
2. Įgyvendinkite ``translate_for_database(expansion, database)`` -- BE DKM,
   grynomis taisyklėmis išverskite išraišką į ``pubmed`` / ``openalex`` /
   ``scopus`` sintaksę.
3. Paimkite SAVO pradinę užklausą, išplėskite ją, išverskite į visas tris
   duomenų bazes ir palyginkite. Vienu sakiniu įvardykite reikšmingiausią
   skirtumą tarp bazių.
4. Prisiminkite: DKM pasiūlyta užklausa VISADA reikalauja nepriklausomo
   žmogaus įvertinimo (skyriaus teiginys C07).

Paleidimas: ``python -m examples.chapter.s04_query_expansion.exercise``
```

1. Paleiskite paruošimo langelį. 2. Užpildykite `TODO` funkcijas pratimo langelyje ir paleiskite jį. 3. Palyginkite su pavyzdiniu sprendimu.

In [ ]:

from __future__ import annotations

from typing import Literal

from pydantic import BaseModel, ConfigDict, StrictStr

from examples.chapter import toolkit


class Concept(BaseModel):
    model_config = ConfigDict(strict=True)

    label: StrictStr
    terms: list[StrictStr]


class QueryExpansion(BaseModel):
    model_config = ConfigDict(strict=True)

    concepts: list[Concept]
    boolean_query: StrictStr


def expand_query(seed: str, *, call_llm=toolkit.call_llm) -> QueryExpansion:
    """TODO: paklauskite DKM ir patikrinkite atsakymą."""
    raise NotImplementedError


def translate_for_database(
    expansion: QueryExpansion, database: Literal["pubmed", "openalex", "scopus"]
) -> str:
    """TODO: grynos taisyklės, jokio DKM."""
    raise NotImplementedError


def main() -> None:
    seed = "large language models for systematic review screening"
    scripted = toolkit.scripted_llm(
        {seed: toolkit.SCRIPTED_RESPONSES["s04"][seed]},
        key_fn=lambda messages: messages[-1]["content"],
    )
    print("S4 query expansion exercise")
    try:
        expansion = expand_query(seed, call_llm=scripted)
    except NotImplementedError:
        print("expand_query() is not implemented yet -- fill it in and re-run.")
        return
    print(f"concepts: {[c.label for c in expansion.concepts]}")
    for db in ("pubmed", "openalex", "scopus"):
        try:
            print(f"[{db}] {translate_for_database(expansion, db)}")
        except NotImplementedError:
            print(f"[{db}] translate_for_database() not implemented yet")


main()

## Pavyzdinis sprendimas

Pažiūrėkite tik pabandę patys. Žemiau esantys testai tikrina **pavyzdinį sprendimą**, ne jūsų kodą.

In [ ]:
!cat examples/chapter/s04_query_expansion/solution.py

In [ ]:
!python -m pytest examples/chapter/s04_query_expansion -q

In [ ]:
# NEPRIVALOMA: tikri kreipiniai i DKM (sumoka uz naudojimą). Raktą laikykite
# Colab "Secrets" (rakto piktograma kairėje), niekada ne užrašykite kode.
# from google.colab import userdata
# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")